In [4]:
import pandas as pd
import numpy as np
import yaml
import re
import os
from itertools import permutations
DB_FOLDER = f"dataset\database_v3"
GRAPH_FOLDER = os.path.join(DB_FOLDER, "Graph")

In [5]:
node_df = pd.read_csv(os.path.join(GRAPH_FOLDER, "node_id.csv"))
node_df

,node_id,project_id,count
0,0,0,270
1,1,1,253
2,2,2,372
3,3,3,149
4,4,4,65
...,...,...,...
2492,2492,2492,57
2493,2493,2493,1
2494,2494,2494,782
2495,2495,2495,578


In [ ]:
# edges_list = []

# # --------------------------------------------------
# # 1. build edges within same project
# # --------------------------------------------------
# for project_id, group in node_df.groupby('project_id'):
#     node_ids = group['node_id'].tolist()
    
#     # create all ordered pairs (i, j) where i != j
#     for i, j in permutations(node_ids, 2):
#         edges_list.append((i, j))

# # --------------------------------------------------
# # 2. convert to dataframe
# # --------------------------------------------------
# edges_df = pd.DataFrame(edges_list, columns=['node_id', 'neig_id'])

# # --------------------------------------------------
# # 3. save
# # --------------------------------------------------
# edge_dir = os.path.join(DB_FOLDER, "edges")
# os.makedirs(edge_dir, exist_ok=True)

# edges_df.to_csv(os.path.join(edge_dir, "same_project.csv"), index=False)

In [7]:
threshold = 250  # meters

project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))

# --------------------------------------------------
# 1. keep unique project coordinates
# --------------------------------------------------
proj_coord = (
    project_df[['project_id', 'latitude', 'longitude']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

proj_coord['latitude'] = pd.to_numeric(proj_coord['latitude'], errors='coerce')
proj_coord['longitude'] = pd.to_numeric(proj_coord['longitude'], errors='coerce')
proj_coord = proj_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 2. haversine distance (meters)
# --------------------------------------------------
R = 6371000.0

lat = np.radians(proj_coord['latitude'].to_numpy())
lon = np.radians(proj_coord['longitude'].to_numpy())

dlat = lat[:, None] - lat[None, :]
dlon = lon[:, None] - lon[None, :]

a = np.sin(dlat / 2)**2 + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin(dlon / 2)**2
c = 2 * np.arcsin(np.sqrt(a))
dist_mat = R * c

# --------------------------------------------------
# 3. build edges (<= threshold, exclude self)
# --------------------------------------------------
mask = (dist_mat <= threshold) & (dist_mat > 0)

src_idx, dst_idx = np.where(mask)

edges_df = pd.DataFrame({
    'project_id': proj_coord.iloc[src_idx]['project_id'].to_numpy(),
    'neig_project_id': proj_coord.iloc[dst_idx]['project_id'].to_numpy(),
})

# --------------------------------------------------
# 4. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df = edges_df.sort_values(['project_id', 'neig_project_id']).reset_index(drop=True)

edges_df.to_csv(os.path.join(edge_dir, f"dist_{threshold}.csv"), index=False)

In [17]:
radius = 500  # meters

project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))
mrt_df = pd.read_csv(os.path.join(DB_FOLDER, "Rail_Transport.csv"))

# --------------------------------------------------
# 1. keep unique project coordinates
# --------------------------------------------------
proj_coord = (
    project_df[['project_id', 'latitude', 'longitude']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

proj_coord['latitude'] = pd.to_numeric(proj_coord['latitude'], errors='coerce')
proj_coord['longitude'] = pd.to_numeric(proj_coord['longitude'], errors='coerce')
proj_coord = proj_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 2. prepare MRT coordinates
#    adjust these column names if your file uses different names
# --------------------------------------------------
mrt_coord = mrt_df[['latitude', 'longitude']].copy()
mrt_coord['latitude'] = pd.to_numeric(mrt_coord['latitude'], errors='coerce')
mrt_coord['longitude'] = pd.to_numeric(mrt_coord['longitude'], errors='coerce')
mrt_coord = mrt_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 3. haversine distance: project -> MRT
# --------------------------------------------------
R = 6371000.0  # Earth radius in meters

proj_lat = np.radians(proj_coord['latitude'].to_numpy())
proj_lon = np.radians(proj_coord['longitude'].to_numpy())

mrt_lat = np.radians(mrt_coord['latitude'].to_numpy())
mrt_lon = np.radians(mrt_coord['longitude'].to_numpy())

dlat = proj_lat[:, None] - mrt_lat[None, :]
dlon = proj_lon[:, None] - mrt_lon[None, :]

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(proj_lat)[:, None] * np.cos(mrt_lat)[None, :] * np.sin(dlon / 2.0) ** 2
)
c = 2 * np.arcsin(np.sqrt(a))
proj_mrt_dist = R * c  # shape: [num_projects, num_mrt]

# --------------------------------------------------
# 4. membership matrix:
#    within radius of each MRT circle
# --------------------------------------------------
within_circle = proj_mrt_dist <= radius  # bool matrix [P, M]

# --------------------------------------------------
# 5. if two projects share at least one MRT circle, connect them
# --------------------------------------------------
# project-project shared-circle count
shared_circle = within_circle @ within_circle.T  # bool works like 0/1 in matrix multiply

# edge if share >=1 circle, exclude self
edge_mask = (shared_circle > 0)
np.fill_diagonal(edge_mask, False)

src_idx, dst_idx = np.where(edge_mask)

edges_df = pd.DataFrame({
    'project_id': proj_coord.iloc[src_idx]['project_id'].to_numpy(),
    'neig_project_id': proj_coord.iloc[dst_idx]['project_id'].to_numpy(),
})

edges_df = (
    edges_df
    .drop_duplicates()
    .sort_values(['project_id', 'neig_project_id'])
    .reset_index(drop=True)
)

# --------------------------------------------------
# 6. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df.to_csv(os.path.join(edge_dir, f"mrt_cir_{radius}.csv"), index=False)


In [14]:
eps = 2  # meters

project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))
mrt_df = pd.read_csv(os.path.join(DB_FOLDER, "Rail_Transport.csv"))

# --------------------------------------------------
# 1. keep unique project coordinates
# --------------------------------------------------
proj_coord = (
    project_df[['project_id', 'latitude', 'longitude']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

proj_coord['latitude'] = pd.to_numeric(proj_coord['latitude'], errors='coerce')
proj_coord['longitude'] = pd.to_numeric(proj_coord['longitude'], errors='coerce')
proj_coord = proj_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 2. prepare MRT coordinates
# --------------------------------------------------
mrt_coord = mrt_df[['latitude', 'longitude']].copy()
mrt_coord['latitude'] = pd.to_numeric(mrt_coord['latitude'], errors='coerce')
mrt_coord['longitude'] = pd.to_numeric(mrt_coord['longitude'], errors='coerce')
mrt_coord = mrt_coord.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

# --------------------------------------------------
# 3. haversine distance: project -> MRT
# --------------------------------------------------
R = 6371000.0  # Earth radius in meters

proj_lat = np.radians(proj_coord['latitude'].to_numpy())
proj_lon = np.radians(proj_coord['longitude'].to_numpy())

mrt_lat = np.radians(mrt_coord['latitude'].to_numpy())
mrt_lon = np.radians(mrt_coord['longitude'].to_numpy())

dlat = proj_lat[:, None] - mrt_lat[None, :]
dlon = proj_lon[:, None] - mrt_lon[None, :]

a = (
    np.sin(dlat / 2.0) ** 2
    + np.cos(proj_lat)[:, None] * np.cos(mrt_lat)[None, :] * np.sin(dlon / 2.0) ** 2
)
c = 2 * np.arcsin(np.sqrt(a))
proj_mrt_dist = R * c  # shape: [num_projects, num_mrt]

# --------------------------------------------------
# 4. nearest MRT distance for each project
# --------------------------------------------------
nearest_mrt_dist = proj_mrt_dist.min(axis=1)

# --------------------------------------------------
# 5. if two projects have similar nearest-MRT distance, connect them
# --------------------------------------------------
dist_diff = np.abs(nearest_mrt_dist[:, None] - nearest_mrt_dist[None, :])
edge_mask = dist_diff < eps
np.fill_diagonal(edge_mask, False)

src_idx, dst_idx = np.where(edge_mask)

edges_df = pd.DataFrame({
    'project_id': proj_coord.iloc[src_idx]['project_id'].to_numpy(),
    'neig_project_id': proj_coord.iloc[dst_idx]['project_id'].to_numpy(),
})

edges_df = (
    edges_df
    .drop_duplicates()
    .sort_values(['project_id', 'neig_project_id'])
    .reset_index(drop=True)
)

# --------------------------------------------------
# 6. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df.to_csv(os.path.join(edge_dir, f"mrt_nearest_dist_eps_{eps}.csv"), index=False)


In [15]:
project_df = pd.read_csv(os.path.join(DB_FOLDER, "project.csv"))

# --------------------------------------------------
# 1. keep projects with non-empty Condo_Age_2026
# --------------------------------------------------
project_age = (
    project_df[['project_id', 'Condo_Age_2026']]
    .drop_duplicates(subset=['project_id'])
    .copy()
)

age_text = project_age['Condo_Age_2026'].astype('string').str.strip()
project_age = project_age[age_text.notna() & (age_text != '')].copy()
project_age['Condo_Age_2026'] = age_text[project_age.index]

# --------------------------------------------------
# 2. connect projects with the same Condo_Age_2026
# --------------------------------------------------
edges_list = []

for _, group in project_age.groupby('Condo_Age_2026'):
    project_ids = group['project_id'].drop_duplicates().to_numpy()
    if len(project_ids) < 2:
        continue

    for project_id, neig_project_id in permutations(project_ids, 2):
        edges_list.append((project_id, neig_project_id))

edges_df = pd.DataFrame(edges_list, columns=['project_id', 'neig_project_id'])

edges_df = (
    edges_df
    .drop_duplicates()
    .sort_values(['project_id', 'neig_project_id'])
    .reset_index(drop=True)
)

# --------------------------------------------------
# 3. save
# --------------------------------------------------
edge_dir = os.path.join(GRAPH_FOLDER, "edges")
os.makedirs(edge_dir, exist_ok=True)

edges_df.to_csv(os.path.join(edge_dir, "same_condo_age_2026.csv"), index=False)
